
# PHASE 3 — DATA CLEANING & INTEGRATION
## Borewell Sentinel India

**Phase:** 3  
**Status:** COMPLETED  
**Input:** 31 original CGWB state/UT CSV datasets  
**Total source records:** 174,635

---

## 1. Phase Objective

The objective of this phase is to transform the collected CGWB groundwater chemical monitoring datasets into a unified, structured and traceable analytical dataset suitable for further exploratory analysis and machine learning preparation.

The original source files must remain unchanged throughout the process.

All cleaning and integration operations are performed on a separate working copy.

## 2. Phase Workflow

This phase consists of four major batches.

### Batch 1 — Dataset Integration and Type Conversion

- Load and integrate all 31 original CSV files.
- Verify schema consistency.
- Preserve source filename and original row references.
- Standardize blank and known placeholder values.
- Convert geographical and chemical measurement columns into numeric types.
- Parse sampling dates.
- Identify malformed numeric values.

### Batch 2 — Chemical Measurement Quality Control

- Identify negative chemical measurements.
- Flag zero measurements for investigation.
- Identify invalid pH readings.
- Detect statistical outliers using the Interquartile Range (IQR) method.
- Generate parameter-wise quality summaries.
- Preserve suspicious measurements rather than automatically deleting them.

### Batch 3 — Station Identity and Duplicate Handling

- Standardize station, state and district labels in separate fields.
- Create provisional station identity identifiers.
- Generate coordinate-based location fingerprints.
- Investigate repeated station-date observations.
- Classify repeated observations according to coordinate and chemical measurement differences.
- Preserve all original observations.

### Batch 4 — Final Validation and Export

- Assemble the integrated processed master dataset.
- Attach measurement quality flags.
- Generate station-location candidate reports.
- Generate repeated station-date audit reports.
- Export chemical quality summaries and outlier examples.
- Verify record preservation and export integrity.

## 3. Data Preservation Principles

The following principles govern this phase:

1. Never modify the original extracted CSV files.
2. Never automatically delete records because of statistical outliers.
3. Never replace legitimate zero measurements with missing values.
4. Never merge station records based only on matching names.
5. Never treat repeated station-date observations as duplicates without supporting evidence.
6. Preserve original administrative names and codes.
7. Maintain source traceability for every processed record.
8. Document quality problems through explicit flags.

## 4. Expected Outputs

The processed dataset and supporting audit files will be stored in:

`data/processed/`

The master dataset will serve as the unified analytical foundation for subsequent project phases.

---


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

EXTRACTED_DIR = PROJECT_ROOT / "data" / "extracted"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

csv_files = sorted(EXTRACTED_DIR.rglob("*.csv"))

raw_datasets = {}
original_columns = None

for file in csv_files:

    df = pd.read_csv(
        file,
        dtype=str,
        keep_default_na=False,
        na_filter=False
    )

    if original_columns is None:
        original_columns = list(df.columns)

    elif list(df.columns) != original_columns:
        raise ValueError(f"Schema mismatch detected: {file.name}")

    df["_source_file"] = file.stem
    df["_source_row"] = np.arange(len(df))

    raw_datasets[file.stem] = df

raw_master_df = pd.concat(
    raw_datasets.values(),
    ignore_index=True
)

print("RAW MASTER DATASET CREATED")
print("=" * 50)
print("Source files:", len(raw_datasets))
print("Total records:", len(raw_master_df))
print("Original columns:", len(original_columns))
print("Total columns including source tracking:",
      len(raw_master_df.columns))

RAW MASTER DATASET CREATED
Source files: 31
Total records: 174635
Original columns: 46
Total columns including source tracking: 48


In [2]:
clean_df = raw_master_df.copy()

# Strip leading and trailing whitespace in the working copy.
# The original values remain available in raw_master_df.

for column in original_columns:

    values = clean_df[column].astype("string").str.strip()

    clean_df[column] = values.mask(
        values.eq(""),
        pd.NA
    )

# Columns where the audit identified "-" as a missing-value marker.

placeholder_columns = [
    "River",
    "Basin",
    "Tributary",
    "Subtributary",
    "SubSubtributary",
    "Local River",
    "Block",
    "Village",
    "Tehsil"
]

placeholder_summary = {}

for column in placeholder_columns:

    placeholder_mask = clean_df[column].eq("-").fillna(False)

    placeholder_summary[column] = int(
        placeholder_mask.sum()
    )

    clean_df.loc[placeholder_mask, column] = pd.NA

print("MISSING-VALUE STANDARDIZATION")
print("=" * 50)

print("Blank cells converted to missing values:",
      clean_df[original_columns].isna().sum().sum())

print("\nPlaceholder values converted:")
display(
    pd.DataFrame(
        list(placeholder_summary.items()),
        columns=["Column", "Replaced_Placeholders"]
    )
)

MISSING-VALUE STANDARDIZATION
Blank cells converted to missing values: 4088310

Placeholder values converted:


,Column,Replaced_Placeholders
0,River,174635
1,Basin,174635
2,Tributary,174635
3,Subtributary,174635
4,SubSubtributary,174635
5,Local River,174635
6,Block,4749
7,Village,2453
8,Tehsil,1224


In [3]:
# Geographic and chemical measurement columns

measurement_columns = original_columns[19:]

numeric_columns = [
    "Latitude",
    "Longitude"
] + measurement_columns

invalid_numeric_flags = pd.DataFrame(
    index=clean_df.index
)

for column in numeric_columns:

    parsed_values = pd.to_numeric(
        clean_df[column],
        errors="coerce"
    )

    invalid_mask = (
        clean_df[column].notna()
        & parsed_values.isna()
    )

    invalid_numeric_flags[column] = invalid_mask

    clean_df[column] = parsed_values

# Record how many numeric parsing problems each row contains.

clean_df["_invalid_numeric_count"] = (
    invalid_numeric_flags.sum(axis=1)
)

clean_df["_invalid_numeric_columns"] = (
    invalid_numeric_flags.apply(
        lambda row: ", ".join(
            row.index[row].tolist()
        ),
        axis=1
    )
)

# Parse dates without overwriting the original date strings.

clean_df["_parsed_date"] = pd.to_datetime(
    clean_df["Data Acquisition Time"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

print("TYPE CONVERSION COMPLETED")
print("=" * 50)

print("Numeric columns converted:", len(numeric_columns))
print("Invalid numeric measurements:",
      invalid_numeric_flags.sum().sum())

print("Rows containing invalid numeric measurements:",
      (clean_df["_invalid_numeric_count"] > 0).sum())

print("Invalid dates:",
      clean_df["_parsed_date"].isna().sum())

TYPE CONVERSION COMPLETED
Numeric columns converted: 29
Invalid numeric measurements: 6
Rows containing invalid numeric measurements: 4
Invalid dates: 0


In [6]:
print("PHASE 3 — BATCH 1 VALIDATION")
print("=" * 60)

print("\nDATASET DIMENSIONS")
print("Raw records:", len(raw_master_df))
print("Processed working records:", len(clean_df))
print("Original columns:", len(original_columns))
print("Total working columns:", len(clean_df.columns))

print("\nSOURCE FILE INTEGRITY")
print("Files:", raw_master_df["_source_file"].nunique())

print("\nNUMERIC PARSING ISSUES")

numeric_issue_summary = pd.DataFrame({
    "Column": numeric_columns,
    "Invalid_Values": [
        int(invalid_numeric_flags[column].sum())
        for column in numeric_columns
    ],
    "Missing_After_Conversion": [
        int(clean_df[column].isna().sum())
        for column in numeric_columns
    ]
})

display(
    numeric_issue_summary[
        (numeric_issue_summary["Invalid_Values"] > 0)
        | (numeric_issue_summary["Missing_After_Conversion"] > 0)
    ].reset_index(drop=True)
)

print("\nCHEMICAL DATA COMPLETENESS")

chemical_missing_summary = pd.DataFrame({
    "Parameter": measurement_columns,
    "Available_Values": [
        int(clean_df[column].notna().sum())
        for column in measurement_columns
    ],
    "Missing_Values": [
        int(clean_df[column].isna().sum())
        for column in measurement_columns
    ],
    "Missing_Percentage": [
        round(clean_df[column].isna().mean() * 100, 2)
        for column in measurement_columns
    ]
})

display(
    chemical_missing_summary.sort_values(
        "Missing_Percentage",
        ascending=False
    ).reset_index(drop=True)
)

print("\nFINAL CHECKS")
print("Original raw dataset unchanged:",
      len(raw_master_df) == len(clean_df))

print("All dates parsed:",
      clean_df["_parsed_date"].notna().all())

print("All numeric columns have numeric dtypes:",
      all(
          pd.api.types.is_numeric_dtype(clean_df[column])
          for column in numeric_columns
      ))

print("\nBATCH 1 COMPLETED")

PHASE 3 — BATCH 1 VALIDATION

DATASET DIMENSIONS
Raw records: 174635
Processed working records: 174635
Original columns: 46
Total working columns: 51

SOURCE FILE INTEGRITY
Files: 31

NUMERIC PARSING ISSUES


,Column,Invalid_Values,Missing_After_Conversion
0,Latitude,0,152
1,Longitude,0,152
2,Potential of Hydrogen (pH),4,11906
3,Electric Conductivity (μS/cm),0,1606
4,Total Dissolved Solids (mg/L),0,144484
5,Carbonate (mg/L),0,54406
6,Bicarbonate (mg/L),0,27464
7,Total Alkalinity (mg/L as CaCO3),1,94699
8,Chloride (mg/L),0,12601
9,Nitrate N (mgN/L),0,174635



CHEMICAL DATA COMPLETENESS


,Parameter,Available_Values,Missing_Values,Missing_Percentage
0,Nitrate N (mgN/L),0,174635,100.00
1,Potassium (mg/L),0,174635,100.00
2,Fluoride (mg/L),0,174635,100.00
3,Lead (mg/L),794,173841,99.55
4,Cadmium (mg/L),797,173838,99.54
5,Copper (mg/L),796,173839,99.54
6,Zinc (mg/L),796,173839,99.54
7,Nickel (mg/L),797,173838,99.54
8,Manganese (mg/L),797,173838,99.54
9,Chromium (mg/L),797,173838,99.54



FINAL CHECKS
Original raw dataset unchanged: True
All dates parsed: True
All numeric columns have numeric dtypes: True

BATCH 1 COMPLETED


In [7]:
quality_flags = pd.DataFrame(index=clean_df.index)

for column in measurement_columns:

    values = clean_df[column]

    # Negative concentration or measurement
    quality_flags[f"{column}__negative"] = values < 0

    # Recorded zero: flagged for review, not automatically treated as missing
    quality_flags[f"{column}__zero"] = values == 0

    # Non-finite numeric values
    quality_flags[f"{column}__nonfinite"] = (
        values.notna() & ~np.isfinite(values)
    )

    # Invalid text-to-number conversions
    quality_flags[f"{column}__invalid_parse"] = (
        invalid_numeric_flags[column]
    )

# Special pH screening
ph_values = clean_df["Potential of Hydrogen (pH)"]

quality_flags["pH__outside_0_14"] = (
    (ph_values < 0) | (ph_values > 14)
)

quality_flags["pH__zero_suspicious"] = ph_values == 0

print("MEASUREMENT FLAGS CREATED")
print("Records:", len(quality_flags))
print("Quality flag columns:", len(quality_flags.columns))
print("Total negative measurements:",
      sum(
          quality_flags[f"{col}__negative"].sum()
          for col in measurement_columns
      ))
print("Invalid numeric conversions:",
      quality_flags.filter(like="__invalid_parse").sum().sum())

MEASUREMENT FLAGS CREATED
Records: 174635
Quality flag columns: 110
Total negative measurements: 138
Invalid numeric conversions: 6


In [8]:
iqr_results = []
iqr_flags = pd.DataFrame(index=clean_df.index)

for column in measurement_columns:

    values = clean_df[column]

    valid_mask = (
        values.notna()
        & np.isfinite(values)
        & (values >= 0)
    )

    # Exclude pH values outside the physically valid screening range
    if column == "Potential of Hydrogen (pH)":
        valid_mask &= values.between(0, 14)

    valid_values = values[valid_mask]

    if len(valid_values) < 4:
        iqr_flags[column] = False
        continue

    q1 = valid_values.quantile(0.25)
    q3 = valid_values.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    column_flags = (
        valid_mask
        & ((values < lower_bound) | (values > upper_bound))
    )

    iqr_flags[column] = column_flags

    iqr_results.append({
        "Parameter": column,
        "Available_Values": len(valid_values),
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower_Bound": lower_bound,
        "Upper_Bound": upper_bound,
        "IQR_Outlier_Count": int(column_flags.sum())
    })

iqr_summary = pd.DataFrame(iqr_results)

print("IQR OUTLIER DETECTION COMPLETED")
print("Parameters evaluated:", len(iqr_summary))
print("Total statistical outlier flags:",
      int(iqr_summary["IQR_Outlier_Count"].sum()))

IQR OUTLIER DETECTION COMPLETED
Parameters evaluated: 24
Total statistical outlier flags: 124521


In [9]:
quality_summary = []

for column in measurement_columns:

    values = clean_df[column]

    quality_summary.append({
        "Parameter": column,
        "Available": int(values.notna().sum()),
        "Missing": int(values.isna().sum()),
        "Invalid_Parse": int(
            quality_flags[f"{column}__invalid_parse"].sum()
        ),
        "Negative": int(
            quality_flags[f"{column}__negative"].sum()
        ),
        "Zero": int(
            quality_flags[f"{column}__zero"].sum()
        ),
        "IQR_Outliers": int(
            iqr_flags[column].sum()
        ) if column in iqr_flags.columns else 0
    })

quality_summary_df = pd.DataFrame(quality_summary)

display(
    quality_summary_df.sort_values(
        ["IQR_Outliers", "Negative"],
        ascending=False
    ).reset_index(drop=True)
)

print("\nADDITIONAL pH FLAGS")
print("pH outside 0–14:",
      int(quality_flags["pH__outside_0_14"].sum()))

print("pH zero values:",
      int(quality_flags["pH__zero_suspicious"].sum()))


,Parameter,Available,Missing,Invalid_Parse,Negative,Zero,IQR_Outliers
0,Chloride (mg/L),162034,12601,0,1,10,16686
1,Carbonate (mg/L),120229,54406,0,0,104729,15500
2,Sodium (mg/L),143470,31165,0,0,108,15111
3,Electric Conductivity (μS/cm),173029,1606,0,0,2,14015
4,Sulphate (mg/L),126328,48307,0,2,5806,13413
5,Magnesium (mg/L),150422,24213,0,10,162,10007
6,Calcium (mg/L),149855,24780,0,0,76,9282
7,Total Hardness (mgCaCO3/L),144408,30227,0,0,151,9228
8,Iron(mg/L),38702,135933,1,8,9589,4609
9,Bicarbonate (mg/L),147171,27464,0,1,739,4442



ADDITIONAL pH FLAGS
pH outside 0–14: 6
pH zero values: 4


In [10]:
outlier_records = []

for column in measurement_columns:

    if column not in iqr_flags.columns:
        continue

    flagged_indices = iqr_flags.index[iqr_flags[column]]

    if len(flagged_indices) == 0:
        continue

    selected = (
        clean_df.loc[flagged_indices]
        .sort_values(column, ascending=False)
        .head(3)
    )

    for index, row in selected.iterrows():

        outlier_records.append({
            "Parameter": column,
            "Value": row[column],
            "Station": row["Station"],
            "State": row["State"],
            "District": row["District"],
            "Date": row["Data Acquisition Time"],
            "Source_File": row["_source_file"],
            "Source_Row": row["_source_row"]
        })

outlier_records_df = pd.DataFrame(outlier_records)

print("TOP STATISTICAL OUTLIER EXAMPLES")
display(outlier_records_df)

print("\nQUALITY FLAGS ARE INVESTIGATIVE ONLY.")
print("No records or measurements have been deleted or changed.")

TOP STATISTICAL OUTLIER EXAMPLES


,Parameter,Value,Station,State,District,Date,Source_File,Source_Row
0,Potential of Hydrogen (pH),1.113000e+01,Vengaiahpalem,Andhra Pradesh,PRAKASAM,25-05-2001 00:00,gwq_chemical_parameter_manual_cgwb_ap_1961_2025,10448
1,Potential of Hydrogen (pH),1.099000e+01,SATUR,Rajasthan,BUNDI,25-05-2002 00:00,gwq_chemical_parameter_manual_cgwb_rj_1961_2025,11179
2,Potential of Hydrogen (pH),1.092000e+01,Mysorappan Palya,Karnataka,Tumkur,15-05-2022 17:13,gwq_chemical_parameter_manual_cgwb_ka_1961_2025,9149
3,Electric Conductivity (μS/cm),5.700676e+07,Ramanathapuram2,Tamil Nadu,Ramanathapuram,25-05-2003 00:00,gwq_chemical_parameter_manual_cgwb_tn_1961_2025,5484
4,Electric Conductivity (μS/cm),1.976287e+07,Dharapuram2,Tamil Nadu,Erode,25-05-2001 00:00,gwq_chemical_parameter_manual_cgwb_tn_1961_2025,1360
...,...,...,...,...,...,...,...,...
66,Cadmium (mg/L),2.000000e-03,Phul,Punjab,BATHINDA,03-05-2023 11:46,gwq_chemical_parameter_manual_cgwb_pb_1961_2025,4288
67,Cadmium (mg/L),1.000000e-03,Dalla,Punjab,KAPURTHALA,05-05-2023 15:41,gwq_chemical_parameter_manual_cgwb_pb_1961_2025,1472
68,Chromium (mg/L),5.000000e-02,Uchana,Haryana,JIND,14-06-2022 14:45,gwq_chemical_parameter_manual_cgwb_hr_1961_2025,6895
69,Chromium (mg/L),3.000000e-02,Kathana-Pz,Haryana,KAITHAL,12-06-2022 09:51,gwq_chemical_parameter_manual_cgwb_hr_1961_2025,3374



QUALITY FLAGS ARE INVESTIGATIVE ONLY.
No records or measurements have been deleted or changed.


In [11]:
# Normalize labels in separate audit columns

for column, new_column in [
    ("Station", "_station_norm"),
    ("State", "_state_norm"),
    ("District", "_district_norm")
]:
    clean_df[new_column] = (
        clean_df[column]
        .astype("string")
        .str.normalize("NFKC")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.lower()
    )

# Provisional station identity
clean_df["_provisional_station_id"] = (
    clean_df["_state_norm"].fillna("<missing>")
    + "|"
    + clean_df["_district_norm"].fillna("<missing>")
    + "|"
    + clean_df["_station_norm"].fillna("<missing>")
)

# Stable record reference back to its source file and row
clean_df["_record_id"] = (
    clean_df["_source_file"].astype(str)
    + "::"
    + clean_df["_source_row"].astype(str)
)

clean_df["_date_only"] = clean_df["_parsed_date"].dt.date

print("STATION IDENTITY FIELDS CREATED")
print("Provisional station identities:",
      clean_df["_provisional_station_id"].nunique())
print("Unique source record IDs:", clean_df["_record_id"].nunique())
print("Original station labels preserved:",
      "Station" in clean_df.columns)

STATION IDENTITY FIELDS CREATED
Provisional station identities: 23811
Unique source record IDs: 174635
Original station labels preserved: True


In [12]:
clean_df["_coordinate_fingerprint"] = pd.NA

coordinate_mask = (
    clean_df["Latitude"].notna()
    & clean_df["Longitude"].notna()
)

clean_df.loc[coordinate_mask, "_coordinate_fingerprint"] = (
    clean_df.loc[coordinate_mask, "Latitude"].round(5).astype(str)
    + "|"
    + clean_df.loc[coordinate_mask, "Longitude"].round(5).astype(str)
)

location_candidate_summary = (
    clean_df.groupby("_provisional_station_id")
    .agg(
        Records=("_record_id", "size"),
        Coordinate_Fingerprints=("_coordinate_fingerprint", "nunique"),
        Missing_Coordinates=("_coordinate_fingerprint",
                             lambda x: x.isna().sum())
    )
    .reset_index()
)

multiple_location_candidates = location_candidate_summary[
    location_candidate_summary["Coordinate_Fingerprints"] > 1
].sort_values(
    ["Coordinate_Fingerprints", "Records"],
    ascending=False
)

print("LOCATION IDENTITY ANALYSIS")
print("=" * 50)
print("Provisional station identities:",
      len(location_candidate_summary))
print("Identities with multiple coordinate fingerprints:",
      len(multiple_location_candidates))

display(multiple_location_candidates.head(25))

LOCATION IDENTITY ANALYSIS
Provisional station identities: 23811
Identities with multiple coordinate fingerprints: 381


,_provisional_station_id,Records,Coordinate_Fingerprints,Missing_Coordinates
112,andhra pradesh|anantapur|alampur,39,3,0
2023,bihar|aurangabad|obra,13,3,0
22717,west bengal|howrah|domjur pz,8,3,0
22386,west bengal|bankura|mejia2,3,3,0
3475,chhattisgarh|korba|tuman,28,2,0
12272,maharashtra|gondia|dongargaon,26,2,0
12605,maharashtra|nagpur|panchgaon,23,2,0
11115,madhya pradesh|ratlam|mindli,22,2,0
15889,punjab|hoshiarpur|hazipur,22,2,0
15961,punjab|jalandhar|udhopur,22,2,0


In [13]:
repeated_group_results = []

group_columns = [
    "_provisional_station_id",
    "_date_only"
]

for key, group in clean_df.groupby(
    group_columns,
    dropna=False,
    sort=False
):

    if len(group) <= 1:
        continue

    coordinate_count = group[
        "_coordinate_fingerprint"
    ].nunique()

    chemistry_profiles = len(
        group[measurement_columns].drop_duplicates()
    )

    varying_chemical_columns = [
        column for column in measurement_columns
        if group[column].nunique(dropna=False) > 1
    ]

    if coordinate_count > 1:
        classification = "Multiple coordinate fingerprints"

    elif chemistry_profiles > 1:
        classification = "Same coordinates, different chemistry"

    else:
        classification = "Same coordinates, same chemistry"

    repeated_group_results.append({
        "Provisional_Station_ID": key[0],
        "Date": key[1],
        "Records": len(group),
        "Coordinate_Fingerprints": coordinate_count,
        "Chemical_Profiles": chemistry_profiles,
        "Different_Serial_Numbers": group["SlNo"].nunique(),
        "Classification": classification,
        "Varying_Chemical_Columns": varying_chemical_columns
    })

repeated_station_date_audit = pd.DataFrame(
    repeated_group_results
)

print("REPEATED STATION-DATE GROUPS CLASSIFIED:",
      len(repeated_station_date_audit))

REPEATED STATION-DATE GROUPS CLASSIFIED: 295


In [14]:
print("PHASE 3 — BATCH 3 VALIDATION")
print("=" * 60)

print("\nSTATION IDENTITIES")
print("Unique records:", clean_df["_record_id"].nunique())
print("Provisional station identities:",
      clean_df["_provisional_station_id"].nunique())
print("Identities with multiple coordinate fingerprints:",
      len(multiple_location_candidates))

print("\nREPEATED STATION-DATE CLASSIFICATION")

classification_summary = (
    repeated_station_date_audit
    .groupby("Classification")
    .agg(
        Groups=("Records", "size"),
        Records=("Records", "sum")
    )
    .reset_index()
)

display(classification_summary)

print("\nTOP REPEATED STATION-DATE GROUPS")

display(
    repeated_station_date_audit.sort_values(
        ["Records", "Chemical_Profiles"],
        ascending=False
    ).head(20)
)

print("\nEXACT DUPLICATE CHECK")

exact_duplicates = clean_df.duplicated(
    subset=original_columns
).sum()

print("Exact duplicate rows:", exact_duplicates)

print("\nRECORD PRESERVATION")
print("Original integrated record count:", len(raw_master_df))
print("Current working record count:", len(clean_df))
print("Records lost:", len(raw_master_df) - len(clean_df))

print("\nNo records have been deleted or merged.")

PHASE 3 — BATCH 3 VALIDATION

STATION IDENTITIES
Unique records: 174635
Provisional station identities: 23811
Identities with multiple coordinate fingerprints: 381

REPEATED STATION-DATE CLASSIFICATION


,Classification,Groups,Records
0,Multiple coordinate fingerprints,174,349
1,"Same coordinates, different chemistry",79,161
2,"Same coordinates, same chemistry",42,85



TOP REPEATED STATION-DATE GROUPS


,Provisional_Station_ID,Date,Records,Coordinate_Fingerprints,Chemical_Profiles,Different_Serial_Numbers,Classification,Varying_Chemical_Columns
19,andhra pradesh|west godavari|eluru iii,2002-05-25,3,2,3,3,Multiple coordinate fingerprints,"[Potential of Hydrogen (pH), Electric Conducti..."
135,maharashtra|solapur|bhandi-shegaon,2023-05-20,3,1,3,3,"Same coordinates, different chemistry","[Chloride (mg/L), Total Hardness (mgCaCO3/L), ..."
158,maharashtra|sindudurg|insuli,2023-05-20,3,1,3,3,"Same coordinates, different chemistry","[Potential of Hydrogen (pH), Total Dissolved S..."
138,maharashtra|yavatmal|chaparda,2023-05-20,3,1,2,3,"Same coordinates, different chemistry","[Chloride (mg/L), Sodium (mg/L)]"
205,rajasthan|barmer|arniyali,2023-06-30,3,1,1,3,"Same coordinates, same chemistry",[]
0,andhra pradesh|anantapur|alampur,2000-05-25,2,2,2,2,Multiple coordinate fingerprints,"[Potential of Hydrogen (pH), Electric Conducti..."
1,andhra pradesh|anantapur|alampur,2001-05-25,2,2,2,2,Multiple coordinate fingerprints,"[Potential of Hydrogen (pH), Electric Conducti..."
2,andhra pradesh|anantapur|alampur,2002-05-25,2,2,2,2,Multiple coordinate fingerprints,"[Potential of Hydrogen (pH), Electric Conducti..."
3,andhra pradesh|anantapur|alampur,2004-05-25,2,2,2,2,Multiple coordinate fingerprints,"[Potential of Hydrogen (pH), Electric Conducti..."
4,andhra pradesh|anantapur|alampur,2005-05-25,2,2,2,2,Multiple coordinate fingerprints,"[Potential of Hydrogen (pH), Electric Conducti..."



EXACT DUPLICATE CHECK
Exact duplicate rows: 0

RECORD PRESERVATION
Original integrated record count: 174635
Current working record count: 174635
Records lost: 0

No records have been deleted or merged.


In [15]:
processed_df = clean_df.copy()

# Create coordinate-based candidate identifiers.
# These are provisional and do not certify physical well identity.

processed_df["_provisional_location_candidate_id"] = pd.Series(
    pd.NA,
    index=processed_df.index,
    dtype="string"
)

has_coordinates = processed_df["_coordinate_fingerprint"].notna()

processed_df.loc[
    has_coordinates,
    "_provisional_location_candidate_id"
] = (
    processed_df.loc[
        has_coordinates, "_provisional_station_id"
    ].astype(str)
    + "|site|"
    + processed_df.loc[
        has_coordinates, "_coordinate_fingerprint"
    ].astype(str)
)

# Attach measurement quality flags.
quality_flag_export = (
    quality_flags
    .fillna(False)
    .astype(bool)
    .add_prefix("_qc__")
)

# Attach statistical outlier flags.
iqr_flag_export = (
    iqr_flags
    .fillna(False)
    .astype(bool)
    .add_prefix("_qc__iqr_outlier__")
)

processed_df = pd.concat(
    [processed_df, quality_flag_export, iqr_flag_export],
    axis=1
)

# Row-level quality summary
qc_flag_columns = (
    list(quality_flag_export.columns)
    + list(iqr_flag_export.columns)
)

processed_df["_qc__flag_count"] = (
    processed_df[qc_flag_columns].sum(axis=1)
)

processed_df["_qc__has_any_flag"] = (
    processed_df["_qc__flag_count"] > 0
)

print("FINAL PROCESSED DATASET ASSEMBLED")
print("=" * 55)
print("Records:", len(processed_df))
print("Columns:", len(processed_df.columns))
print("Quality flag columns:", len(qc_flag_columns))
print("Provisional location candidates:",
      processed_df["_provisional_location_candidate_id"].nunique())

FINAL PROCESSED DATASET ASSEMBLED
Records: 174635
Columns: 198
Quality flag columns: 137
Provisional location candidates: 24172


C:\Users\pokep\AppData\Local\Temp\ipykernel_13544\388827448.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  processed_df["_qc__flag_count"] = (
C:\Users\pokep\AppData\Local\Temp\ipykernel_13544\388827448.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  processed_df["_qc__has_any_flag"] = (


In [16]:
# Summarize coordinate-specific candidates
location_candidate_report = (
    processed_df.loc[has_coordinates]
    .groupby(
        ["_provisional_station_id", "_coordinate_fingerprint"],
        dropna=False
    )
    .agg(
        Station=("Station", "first"),
        State=("State", "first"),
        District=("District", "first"),
        Latitude=("Latitude", "median"),
        Longitude=("Longitude", "median"),
        Records=("_record_id", "size"),
        Unique_Dates=("_date_only", "nunique"),
        First_Sampled=("_parsed_date", "min"),
        Last_Sampled=("_parsed_date", "max")
    )
    .reset_index()
)

location_candidate_report[
    "_provisional_location_candidate_id"
] = (
    location_candidate_report["_provisional_station_id"]
    + "|site|"
    + location_candidate_report["_coordinate_fingerprint"]
)

classification_summary = (
    repeated_station_date_audit
    .groupby("Classification")
    .agg(
        Groups=("Records", "size"),
        Records=("Records", "sum")
    )
    .reset_index()
)

print("SUPPORTING REPORTS PREPARED")
print("=" * 50)
print("Location candidate rows:", len(location_candidate_report))
print("Repeated station-date groups:",
      len(repeated_station_date_audit))
print("Chemical parameters audited:", len(quality_summary_df))
print("\nREPEATED OBSERVATION SUMMARY")
display(classification_summary)

print("\nLOCATION CANDIDATE SUMMARY")
print(
    "Unique provisional location candidates:",
    location_candidate_report[
        "_provisional_location_candidate_id"
    ].nunique()
)

SUPPORTING REPORTS PREPARED
Location candidate rows: 24172
Repeated station-date groups: 295
Chemical parameters audited: 27

REPEATED OBSERVATION SUMMARY


,Classification,Groups,Records
0,Multiple coordinate fingerprints,174,349
1,"Same coordinates, different chemistry",79,161
2,"Same coordinates, same chemistry",42,85



LOCATION CANDIDATE SUMMARY
Unique provisional location candidates: 24172


In [17]:
MASTER_PATH = (
    PROCESSED_DIR / "groundwater_master_processed.csv"
)

LOCATION_PATH = (
    PROCESSED_DIR / "station_location_candidates.csv"
)

REPEATED_PATH = (
    PROCESSED_DIR / "repeated_station_date_audit.csv"
)

QUALITY_PATH = (
    PROCESSED_DIR / "chemical_quality_summary.csv"
)

OUTLIER_PATH = (
    PROCESSED_DIR / "chemical_outlier_examples.csv"
)

processed_df.to_csv(
    MASTER_PATH,
    index=False,
    encoding="utf-8-sig",
    na_rep=""
)

location_candidate_report.to_csv(
    LOCATION_PATH,
    index=False,
    encoding="utf-8-sig"
)

repeated_station_date_audit.to_csv(
    REPEATED_PATH,
    index=False,
    encoding="utf-8-sig"
)

quality_summary_df.to_csv(
    QUALITY_PATH,
    index=False,
    encoding="utf-8-sig"
)

outlier_records_df.to_csv(
    OUTLIER_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("EXPORT COMPLETED")
print("=" * 55)

for path in [
    MASTER_PATH,
    LOCATION_PATH,
    REPEATED_PATH,
    QUALITY_PATH,
    OUTLIER_PATH
]:
    print(
        path.name,
        "| Size:",
        round(path.stat().st_size / (1024 ** 2), 2),
        "MB"
    )

EXPORT COMPLETED
groundwater_master_processed.csv | Size: 214.26 MB
station_location_candidates.csv | Size: 4.57 MB
repeated_station_date_audit.csv | Size: 0.06 MB
chemical_quality_summary.csv | Size: 0.0 MB
chemical_outlier_examples.csv | Size: 0.01 MB


In [18]:
export_check = pd.read_csv(
    MASTER_PATH,
    dtype=str,
    keep_default_na=False,
    na_filter=False
)

print("PHASE 3 — FINAL EXPORT VALIDATION")
print("=" * 60)

print("\nMASTER DATASET")
print("Expected records:", len(raw_master_df))
print("Exported records:", len(export_check))
print("Expected columns:", len(processed_df.columns))
print("Exported columns:", len(export_check.columns))

print("\nRECORD INTEGRITY")
print("Unique record IDs:",
      export_check["_record_id"].nunique())

print("Duplicate record IDs:",
      export_check["_record_id"].duplicated().sum())

print("Source files represented:",
      export_check["_source_file"].nunique())

print("Original columns preserved:",
      all(col in export_check.columns for col in original_columns))

print("Quality flags exported:",
      sum(col.startswith("_qc__") for col in export_check.columns))

print("\nFILE EXISTS:", MASTER_PATH.exists())

print("\nFINAL ASSERTIONS")

assert len(export_check) == len(raw_master_df)
assert export_check["_record_id"].nunique() == len(raw_master_df)
assert export_check["_source_file"].nunique() == 31
assert all(col in export_check.columns for col in original_columns)
assert export_check["_parsed_date"].ne("").all()

print("All validation checks passed.")
print("\nPHASE 3 — DATA CLEANING & INTEGRATION COMPLETED")

PHASE 3 — FINAL EXPORT VALIDATION

MASTER DATASET
Expected records: 174635
Exported records: 174635
Expected columns: 198
Exported columns: 198

RECORD INTEGRITY
Unique record IDs: 174635
Duplicate record IDs: 0
Source files represented: 31
Original columns preserved: True
Quality flags exported: 139

FILE EXISTS: True

FINAL ASSERTIONS
All validation checks passed.

PHASE 3 — DATA CLEANING & INTEGRATION COMPLETED



# PHASE 3 — FINAL COMPLETION REPORT
## Data Cleaning & Integration

**Project:** Borewell Sentinel India  
**Phase:** 3  
**Status:** COMPLETED

---

## 1. Final Dataset Overview

The 31 original CGWB datasets were successfully integrated into a unified processed master dataset.

| Metric | Final Result |
|---|---:|
| Source CSV files | 31 |
| Original records | 174,635 |
| Processed records | 174,635 |
| Original columns | 46 |
| Final processed columns | 198 |
| Quality flag columns | 139 |
| Unique record identifiers | 174,635 |
| Duplicate record identifiers | 0 |
| Exact duplicate observations | 0 |
| Records removed | 0 |

All 174,635 source records have been preserved.

## 2. Completed Cleaning Operations

### 2.1 Missing-Value Standardization

- Blank and whitespace-only values were standardized as missing values.
- Known `-` placeholders were converted to missing values in the identified geographical and administrative fields.
- Legitimate zero chemical measurements were preserved.
- No statistical or arbitrary imputation was performed.

### 2.2 Numeric and Date Conversion

- Latitude and Longitude were converted to numeric types.
- All available chemical measurement columns were converted to numeric types.
- The six malformed numeric values identified during the audit were converted to missing values in the working copy.
- Invalid numeric conversions were retained as identifiable quality flags.
- Sampling dates were parsed into a separate datetime field.
- All dates were successfully parsed.

### 2.3 Chemical Measurement Quality Control

The processed dataset contains quality flags for:

- Negative measurements.
- Zero measurements.
- Non-finite values.
- Invalid numeric conversions.
- pH values outside the 0–14 screening range.
- Statistical outliers identified using the IQR method.

The audit identified 138 negative chemical measurements across six parameters.

The IQR method was used as a statistical screening technique. Flagged measurements were not automatically deleted or modified.

### 2.4 Station Identity and Geographic Integration

The following supporting fields were created:

- Normalized station name.
- Normalized state name.
- Normalized district name.
- Provisional station identity.
- Coordinate fingerprint.
- Provisional location candidate identifier.
- Unique source record identifier.

Coordinate fingerprints were generated using latitude and longitude rounded to five decimal places.

These are provisional location candidates and are not verified physical well identifiers.

### 2.5 Repeated Station-Date Investigation

Using normalized provisional station identities and calendar dates, repeated groups were classified as follows:

| Classification | Groups | Records |
|---|---:|---:|
| Multiple coordinate fingerprints | 174 | 349 |
| Same coordinates, different chemistry | 79 | 161 |
| Same coordinates, same chemistry | 42 | 85 |
| **Total** | **295** | **595** |

These groups were retained. No observations were merged or removed.

Note: This grouping uses normalized station identities and therefore differs from the original-label grouping reported in Phase 2.

## 3. Final Exported Files

All processed outputs were successfully exported to:

`data/processed/`

| File | Purpose |
|---|---|
| `groundwater_master_processed.csv` | Unified master dataset with measurement quality flags and source tracking |
| `station_location_candidates.csv` | Provisional station-location candidates |
| `repeated_station_date_audit.csv` | Repeated station-date classification report |
| `chemical_quality_summary.csv` | Parameter-wise measurement quality summary |
| `chemical_outlier_examples.csv` | Representative statistical outlier records |

## 4. Final Integrity Validation

The exported master dataset was reopened and validated.

| Validation | Result |
|---|---|
| Expected records | 174,635 |
| Exported records | 174,635 |
| Expected columns | 198 |
| Exported columns | 198 |
| Unique record IDs | 174,635 |
| Duplicate record IDs | 0 |
| Source files represented | 31 |
| Original columns preserved | Yes |
| Quality flags exported | 139 |
| Export file exists | Yes |

All final assertions passed successfully.

**No source records were lost during integration or export.**

## 5. Known Limitations and Pending Considerations

Although the integrated dataset has been successfully generated, certain scientific and data-quality issues remain unresolved.

1. Some chemical parameters have substantial missingness.
2. Nitrate, fluoride and potassium have no available observations.
3. Suspicious and extreme chemical measurements require further investigation.
4. Some station labels correspond to multiple geographic locations.
5. Administrative codes contain inconsistencies.
6. Provisional location candidates have not been independently verified as permanent monitoring wells.
7. Chemical measurements have not been statistically imputed or corrected.

These issues must be considered during exploratory analysis, feature selection and subsequent modelling.

The exported dataset is a cleaned and integrated analytical foundation, not a declaration that every measurement is scientifically validated or immediately suitable for model training.

## 6. Phase Conclusion

Phase 3 successfully transformed the original collection of 31 CGWB CSV datasets into a unified, traceable processed master dataset containing 174,635 records and 198 columns.

The project now has:

- A unified nationwide dataset.
- Standardized missing-value representations.
- Numeric measurement and date fields.
- Measurement-quality flags.
- Provisional station and location identifiers.
- Supporting data-quality reports.
- Verified record preservation and export integrity.

All original source files remain unchanged.

---

<bold>PHASE 3 — DATA CLEANING & INTEGRATION: COMPLETED</bold>

**Next Phase:**

# PHASE 4 — EXPLORATORY DATA ANALYSIS (EDA)

The next phase will investigate groundwater chemical distributions, geographical patterns, historical trends, parameter relationships and contamination characteristics across the available CGWB observations.
